# Module 15: Multimodal LLMs

**Goal:** Handle more than just text — vision, image generation, audio, video.

**What you'll learn:**
- Vision APIs for image analysis and OCR
- Image generation with GPT Image (`gpt-image-2`)
- Text-to-speech and transcription (`gpt-4o-mini-tts`, `gpt-transcribe`)
- Video generation basics
- Realtime Audio / Voice Agents

**Prerequisites:** Module 00 (API basics), OpenAI API key

---

## 0. Setup

In [ ]:
%pip install -q openai python-dotenv

In [ ]:
import os
import json
import base64
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../../.env")
client = OpenAI()

# Images and audio are outside the course's text-only provider layer (llm/), so this
# page calls OpenAI's SDK directly. Models checked against OpenAI's deprecations page
# on 2026-09-26: dall-e-3 was shut down on 2026-05-12, whisper-1 shuts down 2027-02-26.
VISION_MODEL = "gpt-6-luna"        # an OpenAI chat model from llm/models.json
IMAGE_MODEL = "gpt-image-2"
TTS_MODEL = "gpt-4o-mini-tts"
STT_MODEL = "gpt-transcribe"

CAT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"

def look(prompt: str, image_url: str, **kwargs):
    """One image plus a question, through the Responses API."""
    return client.responses.create(
        model=VISION_MODEL,
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": prompt},
            {"type": "input_image", "image_url": image_url},
        ]}],
        **kwargs,
    )

print("✅ Setup complete")

---
## 1. Vision — Analyze an Image from URL

In [ ]:
response = look("Describe this image in 2 sentences.", CAT_URL, max_output_tokens=150)
print(response.output_text)

---
## 2. OCR — Extract Text from Image

In [ ]:
response = look(
    "Extract all visible text from this image.",
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
    max_output_tokens=200,
)
print(response.output_text)

---
## 3. Image Generation — GPT Image

In [ ]:
response = client.images.generate(
    model=IMAGE_MODEL,
    prompt="A minimalist diagram showing a RAG pipeline: document → embeddings → vector DB → retrieval → LLM → answer. Clean, technical style.",
    size="1024x1024",
    n=1,
)
# GPT Image models return the image itself, base64-encoded, rather than a URL.
image_path = Path("rag_pipeline.png")
image_path.write_bytes(base64.b64decode(response.data[0].b64_json))
print(f"✓ Image saved to {image_path}")

---
## 4. Audio — Text to Speech + Transcription

In [ ]:
# Text-to-Speech
audio_path = Path("test_audio.mp3")
with client.audio.speech.with_streaming_response.create(
    model=TTS_MODEL,
    voice="alloy",
    input="Large language models have revolutionized natural language processing.",
) as tts_response:
    tts_response.stream_to_file(audio_path)
print(f"✓ Audio saved to {audio_path}")

# Transcribe it back
with open(audio_path, "rb") as f:
    transcript = client.audio.transcriptions.create(model=STT_MODEL, file=f)

print(f"Transcript: {transcript.text}")

# Cleanup
audio_path.unlink()

---
## 5. Multimodal Pipeline — Image → Structured Report

In [ ]:
response = look("""Analyze this image and produce a JSON report:
{
  "main_subject": "...",
  "style": "...",
  "colors": ["..."],
  "suggested_caption": "..."
}""", CAT_URL, text={"format": {"type": "json_object"}}, max_output_tokens=300)

report = json.loads(response.output_text)
for key, value in report.items():
    print(f"  {key}: {value}")

---
## When to Use Multimodal

| Scenario | Worth it? | Why |
|----------|-----------|-----|
| Extracting text from PDFs/screenshots | Yes | Vision APIs excel at OCR |
| Describing products from photos | Yes | More accurate than manual tagging |
| Analyzing charts/graphs | Yes | LLMs can read visual data |
| Generating marketing images | Maybe | Quality varies; human review needed |
| Generating marketing videos | Emerging | Improving rapidly |
| Voice-based agent interfaces | Yes | Realtime API enables low-latency voice UX |

**Multimodal is the future — every LLM app will eventually handle more than just text.**

---
## 🧪 Exercises

1. **Screenshot to Code**: Take a UI screenshot, send it to a vision model, ask for HTML/CSS. How close is the output?

2. **Receipt Parser**: Build a function that takes a receipt photo and extracts: store, date, items, total.

3. **Multimodal Pipeline**: Build: (1) transcribe audio with `gpt-transcribe`, (2) summarize, (3) generate an illustration with `gpt-image-2`.